# Newton's Method

Wiki reference for [Newton's method](https://ml-viz-ruby.vercel.app/wiki/newtons-method).

**The idea in one sentence.** Newton's method uses **second-order** (curvature) information —
step $= -H^{-1}\nabla f$ — so it solves a quadratic in a **single step** and converges
**quadratically** (the error roughly squares each step) near the optimum, at the cost of forming
and inverting the Hessian and a tendency to **overshoot** far from the minimum.

We implement Newton's method and gradient descent from scratch, **validate the one-step quadratic
solve and the speed advantage**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  '#ccc',
    'xtick.color':      '#888',
    'ytick.color':      '#888',
    'text.color':       '#eee',
    'grid.color':       '#333',
    'lines.linewidth':  2,
})

np.random.seed(42)

## 1 — The wiki's worked example

Function: $f(x,y) = x^2 + xy + y^2 - 3x$, minimum at $(2,-1)$.

Newton's method from $(0,0)$ reaches $(2,-1)$ in exactly **one step**.

In [ ]:
def f(xy):
    x, y = xy
    return x**2 + x*y + y**2 - 3*x

def grad_f(xy):
    x, y = xy
    return np.array([2*x + y - 3, x + 2*y])

H = np.array([[2., 1.], [1., 2.]])   # constant Hessian (quadratic)
H_inv = np.linalg.inv(H)

x0 = np.array([0., 0.])
g0 = grad_f(x0)
x1 = x0 - H_inv @ g0

print(f"Start:         x = {x0},  f = {f(x0):.3f}")
print(f"Gradient:      ∇f = {g0}")
print(f"H⁻¹:           [[{H_inv[0,0]:.3f}, {H_inv[0,1]:.3f}],")
print(f"                [{H_inv[1,0]:.3f}, {H_inv[1,1]:.3f}]]")
print(f"Newton step:   x₁ = {x1},  f = {f(x1):.3f}")
print(f"Gradient at x₁: {grad_f(x1)}  (should be [0, 0])")

### Validate: Newton solves a quadratic in one step

For a quadratic, the second-order model is *exact*, so one Newton step $x_1 = x_0 - H^{-1}\nabla
f(x_0)$ jumps straight to the minimum — the gradient there is zero. We confirm.

In [ ]:
print(f'gradient norm after one Newton step: {np.linalg.norm(grad_f(x1)):.2e}')
assert np.linalg.norm(grad_f(x1)) < 1e-9, 'Newton reaches the exact minimum of a quadratic in a single step'
print('\n✅ using curvature, Newton solves a quadratic exactly in one step')

## 2 — Newton vs. gradient descent: convergence on the 2D quadratic

In [ ]:
def gradient_descent(grad_fn, x0, lr, n_steps):
    path = [x0.copy()]
    x = x0.copy()
    for _ in range(n_steps):
        x = x - lr * grad_fn(x)
        path.append(x.copy())
    return np.array(path)

def newtons_method(grad_fn, H_inv_fn, x0, n_steps):
    path = [x0.copy()]
    x = x0.copy()
    for _ in range(n_steps):
        x = x - H_inv_fn(x) @ grad_fn(x)
        path.append(x.copy())
    return np.array(path)

x0 = np.array([0., 0.])

path_gd   = gradient_descent(grad_f, x0, lr=0.1, n_steps=50)
path_newt = newtons_method(grad_f, lambda x: H_inv, x0, n_steps=5)

# Loss along each path
loss_gd   = [f(p) for p in path_gd]
loss_newt = [f(p) for p in path_newt]

# Contour plot
xx, yy = np.meshgrid(np.linspace(-0.5, 3, 200), np.linspace(-2, 1, 200))
ZZ = xx**2 + xx*yy + yy**2 - 3*xx

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Trajectories
ax1.contourf(xx, yy, ZZ, levels=25, cmap='viridis', alpha=0.7)
ax1.plot(path_gd[:, 0], path_gd[:, 1], 'o-', color='#f97316',
         markersize=3, label=f'GD (lr=0.1, 50 steps)')
ax1.plot(path_newt[:, 0], path_newt[:, 1], 's-', color='#ef4444',
         markersize=8, label='Newton (1 step!)')
ax1.scatter([2], [-1], color='white', s=100, zorder=5, marker='*', label='minimum (2,-1)')
ax1.set_title('Convergence trajectories')
ax1.legend(fontsize=9)

# Loss curves
ax2.semilogy(loss_gd, color='#f97316', label='Gradient descent')
ax2.semilogy(range(len(loss_newt)), loss_newt, 's-', color='#ef4444',
             markersize=8, label='Newton')
ax2.axhline(f(np.array([2., -1.])), color='#555', linestyle='--', linewidth=1)
ax2.set_xlabel('Step'); ax2.set_ylabel('f(x) − f* (log scale)')
ax2.set_title('Newton reaches minimum in 1 step')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Validate: Newton beats gradient descent

From the same start on the quadratic, Newton reaches the minimum immediately while gradient
descent (first-order) still has a large gradient after one step. We confirm the speed gap.

In [ ]:
start = np.array([5.0, 5.0])
gd_path = gradient_descent(grad_f, start, lr=0.1, n_steps=50)
nt_path = newtons_method(grad_f, lambda x: H_inv, start, n_steps=50)
print(f'|grad| after 1 step: Newton {np.linalg.norm(grad_f(nt_path[1])):.2e}, GD {np.linalg.norm(grad_f(gd_path[1])):.3f}')
assert np.linalg.norm(grad_f(nt_path[1])) < 1e-9, 'Newton reaches the minimum in one step'
assert np.linalg.norm(grad_f(gd_path[1])) > np.linalg.norm(grad_f(nt_path[1])), 'gradient descent needs many steps for the same progress'
print('\n✅ second-order curvature makes Newton far faster than first-order GD (per step)')

## 3 — Newton vs. GD on Rosenbrock (non-quadratic)

Rosenbrock: $f(x,y) = (1-x)^2 + 100(y-x^2)^2$, minimum at $(1,1)$.
GD struggles with the elongated banana shape; Newton's quadratic model
captures the curvature and converges faster.

In [ ]:
def rosenbrock(xy):
    x, y = xy
    return (1 - x)**2 + 100*(y - x**2)**2

def grad_rosenbrock(xy):
    x, y = xy
    dfdx = -2*(1 - x) + 100*2*(y - x**2)*(-2*x)
    dfdy = 100*2*(y - x**2)
    return np.array([dfdx, dfdy])

def hess_rosenbrock(xy):
    x, y = xy
    d2fdx2  = 2 + 100*(-4*y + 12*x**2)
    d2fdxdy = 100*(-4*x)
    d2fdy2  = 100*2
    return np.array([[d2fdx2, d2fdxdy], [d2fdxdy, d2fdy2]])

x0 = np.array([-0.5, 0.5])

path_gd_r = gradient_descent(grad_rosenbrock, x0, lr=1e-3, n_steps=5000)
path_newt_r = newtons_method(grad_rosenbrock,
                              lambda x: np.linalg.inv(hess_rosenbrock(x)),
                              x0, n_steps=20)

loss_gd_r   = [rosenbrock(p) for p in path_gd_r]
loss_newt_r = [rosenbrock(p) for p in path_newt_r]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

xx, yy = np.meshgrid(np.linspace(-1.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
ZZ = (1 - xx)**2 + 100*(yy - xx**2)**2

ax1.contourf(xx, yy, np.log1p(ZZ), levels=30, cmap='viridis', alpha=0.7)
ax1.plot(path_gd_r[:, 0], path_gd_r[:, 1], '-', color='#f97316',
         linewidth=0.8, label='GD (5000 steps)')
ax1.plot(path_newt_r[:, 0], path_newt_r[:, 1], 's-', color='#ef4444',
         markersize=6, label='Newton (20 steps)')
ax1.scatter([1], [1], color='white', s=100, zorder=5, marker='*', label='minimum (1,1)')
ax1.set_title('Rosenbrock trajectories (log contours)')
ax1.legend(fontsize=9)

ax2.semilogy(loss_gd_r, color='#f97316', label='GD (5000 steps)')
ax2.semilogy(loss_newt_r, 's-', color='#ef4444', markersize=6, label='Newton (20 steps)')
ax2.set_xlabel('Step'); ax2.set_ylabel('f(x) (log scale)')
ax2.set_title('Newton converges far faster')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"GD final:    x={path_gd_r[-1].round(4)}, f={loss_gd_r[-1]:.4f}")
print(f"Newton final: x={path_newt_r[-1].round(4)}, f={loss_newt_r[-1]:.6f}")

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **overshoot far from optimum** | the local model is a poor global fit (demo) — damp / line-search |
| **Hessian cost** | forming + inverting $H$ is $O(d^3)$ — quasi-Newton (BFGS) approximates it |
| **non-convexity** | Newton can move toward saddles/maxima — check curvature |
| **singular Hessian** | $H^{-1}$ undefined — regularize (Levenberg-Marquardt) |
| **stochastic settings** | noisy Hessians make pure Newton impractical for deep nets |

Demo: Newton converges quadratically near the optimum but overshoots far from it.

In [ ]:
# Newton's double-edged nature on a NON-quadratic (Rosenbrock): near the optimum it converges
# QUADRATICALLY (the gradient norm collapses super-linearly, digits doubling each step), but FAR
# from the optimum the raw Newton step can OVERSHOOT — the gradient temporarily grows — because
# the local quadratic model is a poor global fit. We show both in one run.
x = np.array([-0.5, 0.5])
gn = [np.linalg.norm(grad_rosenbrock(x))]
for _ in range(12):
    x = x - np.linalg.solve(hess_rosenbrock(x), grad_rosenbrock(x))
    gn.append(np.linalg.norm(grad_rosenbrock(x)))
print('gradient norm per Newton step:', [f'{v:.1e}' for v in gn[:9]])
assert gn[-1] < 1e-6, 'Newton eventually converges quadratically to the minimum'
assert max(gn) > gn[0], 'but far from the optimum a raw Newton step can overshoot (the gradient temporarily grows)'
print('\nNewton is fast near the optimum but fragile far from it -> use a line search / trust region / damping.')

## ✏️ Your turn

### Exercise 1 — Convergence rate

For a strongly convex quadratic, Newton's method converges in **one step**.
For Rosenbrock (non-quadratic), it takes more. Measure the convergence *rate*:
plot $\log \|\nabla f\|$ vs. iteration for Newton's method on Rosenbrock.
Near the minimum, you should see quadratic convergence: the log of the gradient
norm decreases roughly as a straight line when plotted on a doubly-log scale.

In [ ]:
# TODO(you): measure ||∇f|| at each Newton step and plot log(||∇f||) vs. iteration
x = np.array([-0.5, 0.5])
grad_norms = [np.linalg.norm(grad_rosenbrock(x))]

for _ in range(15):
    H = hess_rosenbrock(x)
    x = x - np.linalg.solve(H, grad_rosenbrock(x))
    grad_norms.append(np.linalg.norm(grad_rosenbrock(x)))

# TODO: plot grad_norms on a log scale and observe the convergence shape
# assert grad_norms[-1] < 1e-8, "Did not converge"
# print(f"Converged to x={x.round(6)} in {len(grad_norms)-1} steps")

### Exercise 2 — Diagonal Newton (Adam-like)

Instead of inverting the full Hessian, use only the diagonal: replace $H^{-1}$
with $\text{diag}(H)^{-1}$ (a vector of per-coordinate curvature corrections).
Compare this "diagonal Newton" to full Newton and plain GD on the 2D quadratic.

<details>
<summary>Solution outline</summary>

```python
def diagonal_newton(grad_fn, hess_fn, x0, n_steps):
    path = [x0.copy()]
    x = x0.copy()
    for _ in range(n_steps):
        g = grad_fn(x)
        H_diag = np.diag(hess_fn(x))  # only diagonal entries
        x = x - g / H_diag            # element-wise division
        path.append(x.copy())
    return np.array(path)
# For the 2D quadratic, diagonal Newton still converges in 1 step because
# the cross-terms (off-diagonal Hessian) are smaller than the diagonal.
```
</details>

## Key takeaways

- **Second-order:** step $= -H^{-1}\nabla f$ uses curvature (verified one-step quadratic solve).
- **Faster than GD** per step near the optimum (verified).
- **Quadratic convergence** near the minimum — the gradient collapses super-linearly (demo).
- **Fragile far away:** it can overshoot (demo) — needs damping / a trust region; and the
  Hessian is $O(d^3)$ to invert.